# Section 05 lab — Joins and Parquet output

Run the section setup once. Work through each activity from Step 1 onward; every step explains why it is needed before the code runs.


## Section setup — run once

Create one local Spark session and reuse it throughout this section.


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Inner join

Begin with the join that keeps only keys found on both sides.


### Step 1 — Create transaction and product DataFrames

P999 is intentionally absent from the product reference.


In [ ]:
transactions = spark.createDataFrame([('T1', 'P1'), ('T2', 'P2'), ('T3', 'P999')], ['transaction_id', 'product_id'])
products = spark.createDataFrame([('P1', 'Mouse'), ('P2', 'Keyboard'), ('P3', 'Monitor')], ['product_id', 'product_name'])


### Step 2 — Join matching keys

The shared product_id column is the join key.


In [ ]:
matched = transactions.join(products, 'product_id', 'inner')
matched.show()


### Step 3 — Verify the surviving rows

T3 is removed because P999 has no match.


In [ ]:
print('Matched rows:', matched.count())
assert matched.count() == 2


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
transactions.createOrReplaceTempView('section05_transactions')
products.createOrReplaceTempView('section05_products')
sql_result = spark.sql("""
SELECT t.transaction_id, t.product_id, p.product_name
FROM section05_transactions t
INNER JOIN section05_products p ON t.product_id = p.product_id
ORDER BY t.transaction_id
""")
sql_result.show()
assert sql_result.count() == matched.count()


## Exercise 2 — Left and right joins

Observe which side is preserved when keys do not match.


### Step 1 — Create inputs with unmatched keys on both sides

P999 exists only on the left; P3 exists only on the right.


In [ ]:
transactions = spark.createDataFrame([('T1', 'P1'), ('T2', 'P2'), ('T3', 'P999')], ['transaction_id', 'product_id'])
products = spark.createDataFrame([('P1', 'Mouse'), ('P2', 'Keyboard'), ('P3', 'Monitor')], ['product_id', 'product_name'])


### Step 2 — Preserve the left side

The missing product description for P999 becomes null.


In [ ]:
left_result = transactions.join(products, 'product_id', 'left')
left_result.orderBy("product_id").show()


### Step 3 — Preserve the right side

P3 remains even though no transaction uses it.


In [ ]:
right_result = transactions.join(products, 'product_id', 'right')
right_result.orderBy("product_id").show()
assert left_result.count() == right_result.count() == 3


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
transactions.createOrReplaceTempView('section05_lr_transactions')
products.createOrReplaceTempView('section05_lr_products')
sql_left = spark.sql("""SELECT t.transaction_id, t.product_id, p.product_name FROM section05_lr_transactions t LEFT JOIN section05_lr_products p ON t.product_id = p.product_id ORDER BY t.product_id""")
sql_right = spark.sql("""SELECT t.transaction_id, p.product_id, p.product_name FROM section05_lr_transactions t RIGHT JOIN section05_lr_products p ON t.product_id = p.product_id ORDER BY p.product_id""")
sql_left.show()
sql_right.show()
assert sql_left.count() == left_result.count()
assert sql_right.count() == right_result.count()


## Exercise 3 — Full outer join

Keep unmatched keys from both DataFrames in one result.


### Step 1 — Create the same unmatched inputs

Reusing the shape makes the difference between join types visible.


In [ ]:
transactions = spark.createDataFrame([('T1', 'P1'), ('T2', 'P2'), ('T3', 'P999')], ['transaction_id', 'product_id'])
products = spark.createDataFrame([('P1', 'Mouse'), ('P2', 'Keyboard'), ('P3', 'Monitor')], ['product_id', 'product_name'])


### Step 2 — Run the full outer join

All keys from both sides remain.


In [ ]:
all_keys = transactions.join(products, 'product_id', 'full_outer')
all_keys.orderBy("product_id").show()


### Step 3 — Inspect unmatched rows

Nulls show which side did not contain the key.


In [ ]:
from pyspark.sql import functions as F
unmatched = all_keys.filter(F.col('transaction_id').isNull() | F.col('product_name').isNull())
unmatched.show()
assert unmatched.count() == 2


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
transactions.createOrReplaceTempView('section05_outer_transactions')
products.createOrReplaceTempView('section05_outer_products')
sql_result = spark.sql("""
SELECT COALESCE(t.product_id, p.product_id) AS product_id, t.transaction_id, p.product_name
FROM section05_outer_transactions t
FULL OUTER JOIN section05_outer_products p ON t.product_id = p.product_id
ORDER BY product_id
""")
sql_result.show()
assert sql_result.count() == all_keys.count()


## Exercise 4 — Left-semi and left-anti joins

Return left rows based on whether a match exists, without adding right-side columns.


### Step 1 — Create transactions and known product IDs

Only the key is needed on the right for semi and anti joins.


In [ ]:
transactions = spark.createDataFrame([('T1', 'P1'), ('T2', 'P2'), ('T3', 'P999')], ['transaction_id', 'product_id'])
known_products = spark.createDataFrame([('P1',), ('P2',)], ['product_id'])


### Step 2 — Keep matching left rows

left_semi returns T1 and T2 without adding columns.


In [ ]:
known = transactions.join(known_products, 'product_id', 'left_semi')
known.orderBy("transaction_id").show()


### Step 3 — Keep unmatched left rows

left_anti isolates the unknown product key.


In [ ]:
unknown = transactions.join(known_products, 'product_id', 'left_anti')
unknown.show()
assert unknown.first()['product_id'] == 'P999'


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
transactions.createOrReplaceTempView('section05_semi_transactions')
known_products.createOrReplaceTempView('section05_known_products')
sql_semi = spark.sql("""SELECT t.* FROM section05_semi_transactions t LEFT SEMI JOIN section05_known_products p ON t.product_id = p.product_id ORDER BY transaction_id""")
sql_anti = spark.sql("""SELECT t.* FROM section05_semi_transactions t LEFT ANTI JOIN section05_known_products p ON t.product_id = p.product_id ORDER BY transaction_id""")
sql_semi.show()
sql_anti.show()
assert sql_semi.count() == known.count()
assert sql_anti.count() == unknown.count()


## Exercise 5 — Validate join keys and publish Parquet

Check reference uniqueness before joining, then verify the saved result.


### Step 1 — Create the reference and test uniqueness

Duplicate keys could multiply joined rows.


In [ ]:
from pyspark.sql import functions as F
products = spark.createDataFrame([('P1', 'Mouse'), ('P2', 'Keyboard')], ['product_id', 'product_name'])
duplicate_keys = products.groupBy('product_id').count().filter(F.col('count') > 1)
assert duplicate_keys.count() == 0


### Step 2 — Join the valid reference

The result contains one enriched row per matching transaction.


In [ ]:
transactions = spark.createDataFrame([('T1', 'P1'), ('T2', 'P2')], ['transaction_id', 'product_id'])
enriched = transactions.join(products, 'product_id', 'inner')
enriched.show()


### Step 3 — Write and read the result

Read-back validation checks the published Parquet dataset.


In [ ]:
output_path = '../output/section05_enriched'
enriched.write.mode('overwrite').parquet(output_path)
saved = spark.read.parquet(output_path)
saved.show()
assert saved.count() == enriched.count() == 2


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
products.createOrReplaceTempView('section05_unique_products')
transactions.createOrReplaceTempView('section05_publish_transactions')
spark.sql('SELECT product_id, COUNT(*) AS rows FROM section05_unique_products GROUP BY product_id HAVING COUNT(*) > 1').show()
sql_result = spark.sql("""SELECT t.transaction_id, t.product_id, p.product_name FROM section05_publish_transactions t INNER JOIN section05_unique_products p ON t.product_id = p.product_id ORDER BY t.transaction_id""")
sql_result.show()
assert sql_result.count() == saved.count()
